In [1]:
from pypdf import PdfReader
from IPython.display import display, Markdown
from langchain_text_splitters  import RecursiveCharacterTextSplitter
import os
import torch
from sentence_transformers import SentenceTransformer, util
import chromadb
from openai import OpenAI

%load_ext dotenv
%dotenv ../../05_src/.env
%dotenv ../../05_src/.secrets

In [2]:
USE_GATEWAY = (os.getenv('USE_GATEWAY', 'FALSE').upper() == 'TRUE')

def get_client(use_gateway: bool = USE_GATEWAY) -> OpenAI:
    if use_gateway:
        client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
                    api_key='any value',
                    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})
    else:
        client = OpenAI()
    return client

In [3]:
# read each pdf file, get the text for each page, combine the pages.
# each file is a chapter. combine chapter text into a list of strings

# number of chapters
chapters = 9

keeler = ""

for i in range(chapters):
    chapter = i + 1
    filename = f"../Keeler_Understanding_NMR_Spectroscopy/chapter_{chapter}.pdf"
    reader = PdfReader(filename)

    num_pages = len(reader.pages)

    chapter = ""
    for j in range(num_pages):
        page = reader.pages[j]
        text = page.extract_text()

        keeler += "\n" + text
    
    #keeler.append(chapter)

In [82]:
# combine the above into a function

def read_keeler(filelocation, n_chapters=9):
    keeler = ""
    for i in range(chapters):
        chapter = i + 1
        filename = f"{filelocation}/chapter_{chapter}.pdf"
        reader = PdfReader(filename)
        num_pages = len(reader.pages)

        chapter = ""
        for j in range(num_pages):
            page = reader.pages[j]
            text = page.extract_text()

            keeler += "\n" + text
    print("done")
    return keeler

In [83]:
read_keeler("../Keeler_Understanding_NMR_Spectroscopy")

done


'\nUniversity of Barcelona\nDepartment of Organic Chemistry\nU\uf76e\uf764\uf765\uf772\uf773\uf774\uf761\uf76e\uf764\uf769\uf76e\uf767 NMR\nS\uf770\uf765\uf763\uf774\uf772\uf76f\uf773\uf763\uf76f\uf770\uf779\nJames Keeler\nUniversity of Cambridge, Department of Chemistry\nc⃝James Keeler, 2002 & 2004\n\n1 What this course is about\nThis course is aimed at those who are already familiar with using NMR on a\nday-to-day basis, but who wish to deepen their understanding of how NMR\nexperiments work and the theory behind them. It will be assumed that you are\nfamiliar with the concepts of chemical shifts and couplings, and are used to\ninterpreting proton and 13C spectra. It will also be assumed that you have at\nleast come across simple two-dimensional spectra such as COSY and HMQC\nand perhaps may have used such spectra in the course of your work. Similarly,\nsome familiarity with the nuclear Overhauser eﬀect (NOE) will be assumed.\nThat NMR is a useful for chemists will be taken as self e

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 2800, # chunks of roughly 2800 tokens in length
    chunk_overlap=700, # chunks will overlap a bit, will allow you to see things that would have otherwise been split
    # not necessary if youre doing a semantic split, like for abstract / introduction. those are meant to be separate
    separators=["\n\n", "\n"],
    length_function = len, # specify length function to calculate how many tokens there are (just going by characters)
    add_start_index = True
)

In [6]:
chunks = text_splitter.split_text(keeler)
print(f'Split {len(keeler)} Keeler into {len(chunks)} chunks.' )

Split 463620 Keeler into 222 chunks.


In [7]:
chunks[0]

'University of Barcelona\nDepartment of Organic Chemistry\nU\uf76e\uf764\uf765\uf772\uf773\uf774\uf761\uf76e\uf764\uf769\uf76e\uf767 NMR\nS\uf770\uf765\uf763\uf774\uf772\uf76f\uf773\uf763\uf76f\uf770\uf779\nJames Keeler\nUniversity of Cambridge, Department of Chemistry\nc⃝James Keeler, 2002 & 2004'

In [8]:
chunks[100]

'z-operators are simply spectators.\nFor double- and zero-quantum coherence in which spins i and j are active\nit is convenient to define the following set of operators which represent pure\nmultiple quantum states of given order.  The operators can be expressed in\nterms of the Cartesian or raising and lowering operators.\ndouble quantum,  \nDQ\nDQ\nzero quantum,  \nZQ\np\nII II II II\nII II II II\np\nII II I\nx\nij\nix jx iy jy i j i j\ny\nij\nix jy iy jx i ij ij\nx\nij\nix jx iy jy\n=±\n≡− () ≡+ ()\n≡+ () ≡− ()\n=\n≡+ () ≡\n()\n++ −−\n()\n++ −−\n()\n2\n22\n22\n0\n22\n1\n2\n1\n2\n1\n2\n1\n2\n1\n2\n1\n2 iij ij\ny\nij\niy jx ix jy i ij ij\nII I\nII II II II\n+− −+\n()\n+− −+\n+()\n≡− () ≡− ()ZQ 1\n2\n1\n222\n6.8.2 Evolution of multiple -quantum terms\nEvolution under offsets\nThe double- and zero-quantum operators evolve under offsets in a way\n6–15\nwhich is entirely analogous to the evolution of Ix  and Iy  under free precession\nexcept that the frequencies of evolution are ( Ωi + Ωj

In [9]:
client = get_client()
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

# create embeddings
response = client.embeddings.create(
    input = chunks, 
    model = "text-embedding-3-small"
)


In [10]:
len(response.data)

222

In [11]:
# create a persistent chroma client
chroma_client = chromadb.PersistentClient(path="../chroma_client_assignment_chat")

In [12]:
chroma_client.heartbeat()

1784150586495793000

In [16]:
# create a collection for the data
# only needs to be run once

#collection = chroma_client.create_collection(name = "keeler")

# when running this cell again, there's an error that the collection already exists
# comment out the above code and just get the collection by name instead

collection = chroma_client.get_collection(name="keeler")

In [17]:
# create a list for the embeddings for each chunk
# also create a list of ids. just number the chunks
embeddings = [item.embedding for item in response.data]
ids = [f"id{i}" for i in range(len(chunks))]

In [34]:
import numpy as np
np.shape(embeddings)

(222, 1536)

In [19]:
# add each chunk to the collection
# add the chunk itself, the embeddings, and an id
collection.add(embeddings = embeddings, 
               documents = chunks, 
               ids = ids)

In [68]:
def get_embedding(text, model="text-embedding-3-small"):
    text = text.replace("\n", " ")
    return client.embeddings.create(input=[text], model=model).data[0].embedding

def query_chromadb(query, collection, top_n = 2):
    query_embedding = get_embedding(query)
    results = collection.query(query_embeddings = [query_embedding], n_results = top_n)
    return [(id, score, text) for id, score, text in zip(results['ids'][0], results['distances'][0], results['documents'][0])]

In [69]:
query = "What is dipolar coupling?"

# run a similarity search, this is the principle of RAG
# return the top n chunks
n_chunks = 3
result = query_chromadb(query, collection=collection, top_n=n_chunks)

# result contains (id, score, text)

In [70]:
result

[('id161',
  0.8706774115562439,
  'leading to intra- and inter-molecular relaxation.  Generally, however, nuclei\nin the same molecule can approach much more closely than those in\ndifferent molecules so that intra-molecular relaxation is dominant.\nThe relaxation induced by the dipolar coupling is proportional to the\nsquare of the coupling.  Thus it goes as\nγγ1\n2\n2\n2\n12\n6\n1\nr\nwhere γ1 and γ2 are the gyromagnetic ratios of the two nuclei involved and\nr12 is the distance between them.\nAs the size of the dipolar interaction depends on the product of the\ngyromagnetic ratios of the two nuclei involved, and the resulting relaxation\nrate constants depends on the square of this.  Thus, pairs of nuclei with high\ngyromagnetic ratios are most efficient at promoting relaxation.  For\nexample, every thing else being equal, a proton-proton pair will relax 16\ntimes faster than a carbon-13 proton pair.\nIt is important to realize that in dipolar relaxation the effect is not\nprimaril

In [40]:
result[0][2]

'leading to intra- and inter-molecular relaxation.  Generally, however, nuclei\nin the same molecule can approach much more closely than those in\ndifferent molecules so that intra-molecular relaxation is dominant.\nThe relaxation induced by the dipolar coupling is proportional to the\nsquare of the coupling.  Thus it goes as\nγγ1\n2\n2\n2\n12\n6\n1\nr\nwhere γ1 and γ2 are the gyromagnetic ratios of the two nuclei involved and\nr12 is the distance between them.\nAs the size of the dipolar interaction depends on the product of the\ngyromagnetic ratios of the two nuclei involved, and the resulting relaxation\nrate constants depends on the square of this.  Thus, pairs of nuclei with high\ngyromagnetic ratios are most efficient at promoting relaxation.  For\nexample, every thing else being equal, a proton-proton pair will relax 16\ntimes faster than a carbon-13 proton pair.\nIt is important to realize that in dipolar relaxation the effect is not\nprimarily to distribute the energy from one

In [21]:
chroma_client.list_collections()

[Collection(name=keeler)]

In [42]:
for i, context in enumerate(result):
    print("index", i)
    print("id", context[0])
    print("score", context[1])
    print("text", context[2])
    print("\n")


index 0
id id161
score 0.8706774115562439
text leading to intra- and inter-molecular relaxation.  Generally, however, nuclei
in the same molecule can approach much more closely than those in
different molecules so that intra-molecular relaxation is dominant.
The relaxation induced by the dipolar coupling is proportional to the
square of the coupling.  Thus it goes as
γγ1
2
2
2
12
6
1
r
where γ1 and γ2 are the gyromagnetic ratios of the two nuclei involved and
r12 is the distance between them.
As the size of the dipolar interaction depends on the product of the
gyromagnetic ratios of the two nuclei involved, and the resulting relaxation
rate constants depends on the square of this.  Thus, pairs of nuclei with high
gyromagnetic ratios are most efficient at promoting relaxation.  For
example, every thing else being equal, a proton-proton pair will relax 16
times faster than a carbon-13 proton pair.
It is important to realize that in dipolar relaxation the effect is not
primarily to distri

In [72]:
# put things into functions

def get_context_data(query:str, collection:chromadb.api.models.Collection, top_n:int):
    result = query_chromadb(query, collection, top_n)
    return result

In [73]:
# embellish the response
def generate_prompt(query:str, collection:chromadb.api.models.Collection, top_n:int):
    context_data = get_context_data(query, collection, top_n=top_n)
    prompt = f"Given a query, provide a detailed response using the context from relevant excerpts of James Keeler's book Understanding NMR Spectroscopy.\n\n"
    prompt += f"<query>{query}</query>\n\n"
    prompt += "<context>\n"
    for k, context in enumerate(context_data):
        prompt += f"excerpt id: {context[0]}\n"
        prompt += f"excerpt cosine similarity to query: {context[1]}\n"
        prompt += f"excerpt: {context[2]}\n"
    prompt += "</context>\n\n"
    prompt += "\nBased on the context and nothing else, provide a detailed response to the query."
    return prompt

In [74]:
def generate_response(query:str, collection:chromadb.api.models.Collection, top_n:int=1):
    prompt = generate_prompt(query, collection, top_n)
    #print("Generated Prompt:\n", prompt)
    response = client.responses.create(
        model=MODEL,
        instructions="You are a helpful assistant that provides information based on James Keeler's book Understanding NMR Spectroscopy.",
        input=[{"role": "user", "content": prompt}],
        max_output_tokens=500,
        temperature=0.7
    )
    return response.output_text

In [77]:
response = generate_response("What is an HSQC?", 
                             collection, 
                             top_n=1)

In [78]:
display(Markdown(response))

The Heteronuclear Single Quantum Coherence (HSQC) is a sophisticated NMR experiment designed to correlate hydrogen atoms (typically ^1H) with heteronuclei such as carbon-13 (^13C) or nitrogen-15 (^15N). This technique is particularly valuable in the analysis of large biomolecules, such as proteins, due to its ability to provide detailed information about the connectivity of atoms within a molecular structure.

The HSQC pulse sequence is somewhat more complex than its counterpart, the Heteronuclear Multiple-Bond Correlation (HMQC), yet it offers specific advantages, particularly when embedded in more intricate sequences used for two- and three-dimensional NMR studies of labeled proteins. 

In an HSQC experiment, the sequence generally starts with the application of a 90° pulse on the proton, which generates magnetization that is anti-phase with respect to the coupling to the carbon-13 nucleus. Following this, carbon-13 is subjected to a 90° pulse, converting the proton magnetization into multiple quantum coherence. This step effectively acts as a filter, allowing only the signals from protons that are directly bound to carbon-13 to pass through.

The pulse sequence involves carefully timed delays, denoted as ∆, which are set to 1/(2J12) to ensure optimal coherence and signal evolution. The sequence also includes 180° pulses, which refocus the shifts of the spins involved, enhancing the resolution of the resulting spectrum.

Periods within the sequence, particularly A and C, are designated as spin echoes. During these periods, the application of 180° pulses to both spins refocuses their offsets, while the coupling between them continues to evolve throughout the entire period. This careful orchestration allows for a comprehensive understanding of the spin interactions, leading to observable signals that are crucial for molecular structure elucidation.

In summary, the HSQC is an essential tool in NMR spectroscopy, particularly effective for studying large biological macromolecules by providing a clear picture of the relationships between hydrogen and heteronuclei.

In [ ]:
# this was code from gemini, just for demonstration purposes
# googled "semantic search python" and got this as the AI generated result

import torch
from sentence_transformers import SentenceTransformer, util

# Step 1: Load a pre-trained embedding model
# 'all-MiniLM-L6-v2' maps sentences to a 384-dimensional dense vector space
model = SentenceTransformer("all-MiniLM-L6-v2")

# Step 2: Define your corpus (the documents you want to search through)
corpus = [
    "A fast-moving dog leaps over a hurdle during an agility course.",
    "The chef prepares a classic French pastry with layered dough.",
    "A software engineer writes clean code using Python and Flask.",
    "Several small puppies are playing with a bright red ball.",
    "Building web applications requires solid database knowledge.",
]

# Step 3: Compute embeddings for the corpus
# These vectors encode the semantic meaning of the sentences
corpus_embeddings = model.encode(corpus, convert_to_tensor=True)

# Step 4: Define a natural language search query
query = "canine chasing a toy"

# Step 5: Encode the query and compute similarity scores
query_embedding = model.encode(query, convert_to_tensor=True)

# util.cos_sim computes the cosine similarity between the query and corpus vectors
cosine_scores = util.cos_sim(query_embedding, corpus_embeddings)[0]

# Step 6: Extract and rank the top results
top_k = 2
top_results = torch.topk(cosine_scores, k=top_k)

print(f"Search Query: '{query}'\n")
print("Top Matching Results:")
for score, idx in zip(top_results.values, top_results.indices):
    print(f"- [Score: {score:.4f}] {corpus[idx.item()]}")

# first time running, got a warning:
# Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

# ran once and it worked. prints:
"""
Search Query: 'canine chasing a toy'

Top Matching Results:
- [Score: 0.5408] A fast-moving dog leaps over a hurdle during an agility course.
- [Score: 0.4091] Several small puppies are playing with a bright red ball.
"""

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Search Query: 'canine chasing a toy'

Top Matching Results:
- [Score: 0.5408] A fast-moving dog leaps over a hurdle during an agility course.
- [Score: 0.4091] Several small puppies are playing with a bright red ball.
